# Twitter (microblog)

> **Time-box:** 45–60 minutes. The follow graph and the timeline are where the depth lives — don't spend it on user CRUD.

## Core requirements

1. Users have a unique handle and a display name.
2. A user can post a short tweet (≤ 280 chars).
3. A user can follow / unfollow another user. Following is one-directional.
4. A user can fetch their **home timeline**: tweets from the people they follow, newest first, paginated.
5. A user can fetch any user's **profile timeline**: that user's own tweets.
6. A user can like / unlike a tweet. A tweet shows its like count.

## Stretch goals

- Retweets (a tweet with a reference to an original).
- Replies (threading).
- Mentions of `@handle` in the body — what does the schema need?
- Fan-out: at write-time (push to followers' inboxes) vs read-time (query at read).

## Things the interviewer will probe

- **Pagination:** offset vs cursor (`created_at + id`). What happens with new inserts mid-scroll?
- **Hot follows:** one user has 100M followers. Fan-out on write becomes catastrophic. How do you handle celebrity accounts?
- **Like counts:** denormalized counter on `tweets` vs live `COUNT(*)`? Trade-offs?
- **Composite keys:** `follows` is naturally `(follower_id, followee_id)`. Do you give it a synthetic `id` anyway? Why or why not?
- **Resource naming:** `POST /users/{handle}/follow` vs `POST /follows`? Defend a choice.

---
## Setup

In [ ]:
import json
import sqlite3

import pandas as pd
from fastapi import FastAPI
from fastapi.testclient import TestClient
from IPython.display import display
from pydantic import BaseModel

## Schema

Edit the SQL and re-run this cell to get a fresh in-memory database.

In [ ]:
SCHEMA = """
-- Design your tables here
"""

conn = sqlite3.connect(":memory:", check_same_thread=False)
conn.row_factory = sqlite3.Row
conn.execute("PRAGMA foreign_keys = ON")
conn.executescript(SCHEMA)

print("Tables:", [r[0] for r in conn.execute(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name"
).fetchall()])

## API

> After editing any cell below, re-run from **App** down through **Client**.

In [ ]:
# ── App + models ──────────────────────────────────────────────────────────────
app = FastAPI(title="Twitter")

In [ ]:
# ── Endpoints ─────────────────────────────────────────────────────────────────
@app.get("/healthz")
def healthz():
    return {"status": "ok"}

In [ ]:
# ── Client ────────────────────────────────────────────────────────────────────
client = TestClient(app, raise_server_exceptions=True)
print(client.get("/healthz").json())

## Helpers

In [ ]:
def call(method: str, path: str, **kwargs):
    r = getattr(client, method)(path, **kwargs)
    body = r.json() if r.content else None
    print(f"{method.upper():6s} {path}  →  {r.status_code}")
    if body is not None:
        print(json.dumps(body, indent=2))
    return r


def df(table: str) -> pd.DataFrame:
    return pd.read_sql(f"SELECT * FROM {table}", conn)


def query(sql: str, *params) -> pd.DataFrame:
    return pd.read_sql(sql, conn, params=list(params) if params else None)


def show_all():
    names = [r[0] for r in conn.execute(
        "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name"
    ).fetchall()]
    for name in names:
        count = conn.execute(f"SELECT COUNT(*) FROM {name}").fetchone()[0]
        print(f"\n── {name} ({count} rows) ──")
        display(pd.read_sql(f"SELECT * FROM {name}", conn))

## Demo

In [ ]:
# Call your endpoints here

## All tables

In [ ]:
show_all()